# 🌳 Yggdrasil — Segmentação de LGD

Tutorial do subpacote `yggdrasil.credit_risk.lgd`. Constrói uma régua de **LGD** a partir de uma base sintética, mostra as folhas, a árvore, o `predict` e a régua em PySpark.

**Instalação** (na raiz do repositório):
```bash
pip install -e ".[ui]"   # núcleo + interface interativa
```
> No Databricks: `%pip install -e ".[ui]"` (ou `%pip install optbinning`) num cluster interativo DBR 13.0+ LTS.

In [ ]:
import numpy as np
import pandas as pd
from yggdrasil.credit_risk.lgd import SequentialLGDSegmenter

# --- base sintética: LGD de financiamento de veículo, com DES, OOT e ESTABILIDADE ---
# A população de LTV MIGRA ao longo das safras (dt_ref) — é o que faz o PSI subir nas
# safras mais recentes (OOT e, principalmente, ESTABILIDADE).
rng = np.random.default_rng(42)
def gera(n, t0='2022-01-01'):
    dt_ref = pd.to_datetime(t0) + pd.to_timedelta(rng.integers(0, 365, n), unit='D')
    midx = (dt_ref.year.values - 2022) * 12 + (dt_ref.month.values - 1)   # nº do mês desde jan/2022
    ltv = rng.beta(2.5, 3, n) * 1.4 + 0.3 + 0.008 * midx     # LTV migra ~0,008/mês (drift de população)
    ltv[rng.random(n) < 0.06] = np.nan                       # ~6% de LTV faltante
    atraso = rng.integers(30, 720, n)
    gar = rng.choice(['alienação','aval','fiança','sem garantia'], n, p=[.5,.22,.18,.1])
    lg = {'alienação':0.0,'aval':0.10,'fiança':0.14,'sem garantia':0.28}
    base = (0.05 + 0.45*np.clip(np.nan_to_num(ltv-0.5, nan=0.35), 0, 1)
            + 0.10*(atraso>360) + np.array([lg[g] for g in gar]))
    return pd.DataFrame({'ltv': ltv, 'dias_atraso': atraso, 'garantia': gar, 'dt_ref': dt_ref,
                         'lgd': np.clip(base + rng.normal(0, 0.07, n), 0, 1)})

des = gera(6000, t0='2022-01-01'); des['amostra'] = 'DES'           # desenvolvimento (referência PSI)
oot = gera(2500, t0='2023-07-01'); oot['amostra'] = 'OOT'           # out-of-time (com LGD realizado)
# ESTABILIDADE: público de LGD MAIS RECENTE, só para validar o modelo — SEM variável resposta.
#   Ainda não há LGD realizado para essa safra; serve apenas para PSI/estabilidade da régua e
#   das variáveis de entrada (entra no PSI, mas não tem média de LGD nem entra nas métricas).
est = gera(1800, t0='2024-01-01'); est['amostra'] = 'ESTABILIDADE'
est['lgd'] = np.nan

df = pd.concat([des, oot, est], ignore_index=True)
# resumo por amostra: ESTABILIDADE fica com LGD médio NaN (sem target), mas com volumetria
df.groupby('amostra', sort=False)['lgd'].agg(n='size', lgd_medio='mean')

## LGD ao longo do tempo (safra `dt_ref`)
Análise temporal da base: **LGD médio por mês de referência**. DES (2022) e OOT (2023–24) têm LGD realizado; **ESTABILIDADE** (público mais recente) ainda **não tem target**, então não entra nesta curva — aparece só no PSI. Como a população de LTV migra ao longo das safras, o LGD realizado sobe no tempo.

In [ ]:
import matplotlib.pyplot as plt

# LGD médio e volumetria por safra (mês de dt_ref), por amostra
serie = (df.assign(safra=df['dt_ref'].dt.to_period('M').dt.to_timestamp())
           .groupby(['safra', 'amostra'], observed=True)
           .agg(lgd_medio=('lgd', 'mean'), n=('lgd', 'size'))
           .reset_index())

cores = {'DES': 'steelblue', 'OOT': 'crimson', 'ESTABILIDADE': '#888888'}
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True,
                               gridspec_kw={'height_ratios': [2, 1]})
for am, g in serie.groupby('amostra', observed=True):
    if g['lgd_medio'].notna().any():                 # ESTABILIDADE não tem LGD realizado
        ax1.plot(g['safra'], g['lgd_medio'], marker='o', ms=3.5, lw=2,
                 color=cores.get(am), label=am)
    ax2.bar(g['safra'], g['n'], width=22, color=cores.get(am), alpha=0.6, label=am)
ax1.set_title('LGD médio por safra (dt_ref)', fontweight='bold', color='#15324a')
ax1.set_ylabel('LGD médio'); ax1.grid(alpha=0.2); ax1.legend(title='amostra')
ax2.set_ylabel('volumetria'); ax2.set_xlabel('safra'); ax2.grid(axis='y', alpha=0.2)
fig.tight_layout(); plt.show()

# resumo: LGD médio por ANO e amostra — o LGD realizado sobe no tempo;
# ESTABILIDADE fica NaN (público recente, ainda sem target)
(df.assign(ano=df['dt_ref'].dt.year)
   .pivot_table(index='ano', columns='amostra', values='lgd', observed=True, dropna=False)
   .round(3))

## Instanciar e construir a árvore automática

In [ ]:
seg = SequentialLGDSegmenter(
    df, target='lgd', sample_col='amostra', ref_sample='DES',
    date_col='dt_ref',     # coluna de safra: FORA da modelagem, só p/ análises no tempo
    feature_labels={'ltv':'LTV','dias_atraso':'dias em atraso','garantia':'garantia'},
)
seg.fit_auto(max_depth=3, min_iv=0.02)
seg.suggest_split('root')['msg']

## Folhas finais (nota, LGD, representatividade e PSI)

In [3]:
seg.leaves(with_psi=True)

,segmento,nota_lgd,descricao,profundidade,n,repr_%,lgd_medio,lgd_std,lgd_DES,lgd_OOT,psi_OOT
0,"ltv: (-inf, 1.35] | garantia: {alienação, aval...",1,LTV até 1.35 e garantia em {alienação ou aval ...,3,459,5.4,0.1580,0.0936,0.1532,0.1705,0.0003
1,"ltv: (faltante) | garantia: {alienação, aval, ...",2,LTV faltante e garantia em {alienação ou aval ...,3,243,2.9,0.2629,0.0864,0.2602,0.2684,0.0006
2,"ltv: (-inf, 1.35] | garantia: {alienação, aval...",3,LTV até 1.35 e garantia em {alienação ou aval ...,3,6230,73.3,0.3650,0.1560,0.3481,0.4056,0.0000
3,"ltv: (faltante) | garantia: {alienação, aval, ...",4,LTV faltante e garantia em {alienação ou aval ...,3,220,2.6,0.3712,0.0904,0.3722,0.3691,0.0003
4,ltv: (faltante) | garantia: {sem garantia} | d...,5,LTV faltante e garantia em {sem garantia} e di...,3,32,0.4,0.5103,0.0652,0.5138,0.5014,0.0000
5,"ltv: (-inf, 1.35] | garantia: {sem garantia} |...",6,LTV até 1.35 e garantia em {sem garantia} e di...,3,371,4.4,0.5171,0.1368,0.5041,0.5524,0.0006
6,"ltv: (1.35, inf] | garantia: {alienação, aval,...",7,LTV acima de 1.35 e garantia em {alienação ou ...,3,266,3.1,0.5627,0.0996,0.5281,0.6377,0.0003
7,ltv: (faltante) | garantia: {sem garantia} | d...,8,LTV faltante e garantia em {sem garantia} e di...,3,18,0.2,0.6078,0.0843,0.5919,0.6878,0.0010
8,"ltv: (-inf, 1.35] | garantia: {sem garantia} |...",9,LTV até 1.35 e garantia em {sem garantia} e di...,3,366,4.3,0.6195,0.1422,0.6104,0.6417,0.0000
9,"ltv: (1.35, inf] | garantia: {alienação, aval,...",10,LTV acima de 1.35 e garantia em {alienação ou ...,3,237,2.8,0.6724,0.1040,0.6364,0.7677,0.0003


## Análise de variáveis ao longo do tempo (safra `dt_ref`)
Com `date_col='dt_ref'` no segmenter, as análises temporais já sabem a coluna de safra. Para uma variável **numérica** (LTV): perfil, **PSI atual** por amostra e **PSI por safra** vs DES. Para uma variável **categórica** (garantia): **representatividade (%) de cada categoria por safra** — mostra a distribuição migrando no tempo. A amostra **ESTABILIDADE** (sem LGD) entra só no PSI.

In [ ]:
# Como o seg foi criado com date_col='dt_ref', as análises no tempo dispensam passar a coluna.
# --- variável NUMÉRICA (LTV): perfil + PSI atual + PSI por safra ---
s = seg.variable_summary('ltv')
print(f"LTV — média {s['media']:.3f} · mediana {s['mediana']:.3f} · desvio {s['desvio']:.3f} "
      f"· %missing {s['pct_missing']:.1f}% · IV {s['iv']:.4f} ({s['forca']})")
print("PSI atual por amostra:", {a: round(v, 3) for a, v in s['psi'].items()})
display(seg.variable_psi_by_safra('ltv').tail(6))           # PSI de LTV por safra vs DES

# --- variável CATEGÓRICA (garantia): representatividade de cada categoria por safra ---
display(seg.variable_share_by_safra('garantia').tail(6))    # % de cada categoria por mês

## Árvore em texto

In [4]:
print(seg.tree())

TODA A CARTEIRA  (n=8500, 100.0%, LGD=0.3877)
├─ LTV faltante  (n=513, 6.0%, LGD=0.3369)
│  ├─ garantia em {alienação ou aval ou fiança}  (n=463, 5.4%, LGD=0.3144)
│  │  ├─ garantia em {alienação}  (n=243, 2.9%, LGD=0.2629)  [nota 2]
│  │  └─ garantia em {aval ou fiança}  (n=220, 2.6%, LGD=0.3712)  [nota 4]
│  └─ garantia em {sem garantia}  (n=50, 0.6%, LGD=0.5454)
│     ├─ dias em atraso até 455.5  (n=32, 0.4%, LGD=0.5103)  [nota 5]
│     └─ dias em atraso acima de 455.5  (n=18, 0.2%, LGD=0.6078)  [nota 8]
├─ LTV até 1.35  (n=7426, 87.4%, LGD=0.3724)
│  ├─ garantia em {alienação ou aval ou fiança}  (n=6689, 78.7%, LGD=0.3508)
│  │  ├─ LTV até 0.5278  (n=459, 5.4%, LGD=0.1580)  [nota 1]
│  │  └─ LTV acima de 0.5278  (n=6230, 73.3%, LGD=0.3650)  [nota 3]
│  └─ garantia em {sem garantia}  (n=737, 8.7%, LGD=0.5680)
│     ├─ dias em atraso até 347.5  (n=371, 4.4%, LGD=0.5171)  [nota 6]
│     └─ dias em atraso acima de 347.5  (n=366, 4.3%, LGD=0.6195)  [nota 9]
└─ LTV acima de 1.35  (n=561,

## Aplicar a régua em pandas (`predict`)
Repare que linhas com `ltv` faltante são roteadas para o segmento `(faltante)` — nada é descartado.

In [5]:
novos = gera(800)
pred = seg.predict(novos)
print('cobertura (linhas roteadas):', pred['segmento_lgd'].notna().mean())
pred.head()

cobertura (linhas roteadas): 1.0


,segmento_lgd,nota_lgd,lgd_regua
0,"ltv: (-inf, 1.35] | garantia: {alienação, aval...",3,0.348052
1,"ltv: (1.35, inf] | garantia: {alienação, aval,...",10,0.636384
2,"ltv: (-inf, 1.35] | garantia: {alienação, aval...",3,0.348052
3,"ltv: (-inf, 1.35] | garantia: {alienação, aval...",3,0.348052
4,"ltv: (-inf, 1.35] | garantia: {alienação, aval...",3,0.348052


## Régua em PySpark (scoring em escala no Databricks)

In [6]:
print(seg.to_pyspark())

from pyspark.sql import functions as F

def aplicar_regua_lgd(df, col_seg='segmento_lgd', col_nota='nota_lgd', col_lgd='lgd_regua'):
    """Régua de LGD gerada por SequentialLGDSegmenter (segmento, nota e LGD por folha)."""
    c1 = (F.col("ltv") <= 1.3503407835960388) & F.col("garantia").isin('alienação', 'aval', 'fiança') & (F.col("ltv") <= 0.5277718603610992)
    c2 = F.col("ltv").isNull() & F.col("garantia").isin('alienação', 'aval', 'fiança') & F.col("garantia").isin('alienação')
    c3 = (F.col("ltv") <= 1.3503407835960388) & F.col("garantia").isin('alienação', 'aval', 'fiança') & (F.col("ltv") > 0.5277718603610992)
    c4 = F.col("ltv").isNull() & F.col("garantia").isin('alienação', 'aval', 'fiança') & F.col("garantia").isin('aval', 'fiança')
    c5 = F.col("ltv").isNull() & F.col("garantia").isin('sem garantia') & (F.col("dias_atraso") <= 455.5)
    c6 = (F.col("ltv") <= 1.3503407835960388) & F.col("garantia").isin('sem garantia') & (F.col("dias_atraso") <= 347.5)
    c7 = (F.c

## Interface interativa — workbench em abas (opcional)

No Jupyter/Databricks, dá para construir e validar a árvore clicando. A interface é um **workbench em abas**:

- **① Construir** — cockpit de 3 painéis: à esquerda *qual variável segmentar* (IV contínuo do optbinning + PSI por variável, nos **mesmos bins**) e o histograma de LGD da folha; ao centro o cabeçalho da folha, a árvore e o preview da divisão; à direita as ações, o assistente e a poda. O **Auto-fit** com uma folha selecionada cresce só aquela folha; ao final há um **preview da árvore** em imagem.
- **② Análise de variável** — distribuição, %missing, média/mediana/desvio, faixa de percentis, **PSI atual** e o comportamento por **safra**: numérica → percentis por safra; categórica → **representatividade de cada categoria por safra**.
- **③ Diagnóstico** — folhas (PSI por amostra e teste de hipótese), métricas R² da régua, IC bootstrap e dispersão.
- **④ Validar & Exportar** — monotonicidade, calibração, backtest, exportar `ui.result`, MLflow e Spark.
- **⑤ Histórico** — salvar/carregar a árvore em JSON e a imagem da árvore.

Passe `date_col='dt_ref'` para as análises no tempo. A amostra **ESTABILIDADE** (sem LGD) entra só no **PSI**, não nas médias de LGD nem nas métricas.

```python
from yggdrasil.credit_risk.lgd import LGDSegmenterUI
ui = LGDSegmenterUI(df, target='lgd', sample_col='amostra', ref_sample='DES',
                    date_col='dt_ref',                 # safra p/ as análises no tempo
                    feature_labels={'ltv':'LTV','dias_atraso':'dias em atraso','garantia':'garantia'},
                    tree_samples=['DES','OOT'])         # LGD na árvore (ESTABILIDADE só no PSI)
ui
```

## Registrar no MLflow / Unity Catalog
```python
seg.log_to_mlflow(registered_model_name='catalogo.schema.lgd_segmentacao',
                  registry_uri='databricks-uc')
```